# Replica 충돌 감지기

4개의 라벨된 replica `R1`부터 `R4`가 있고, 순서가 구분되는 요청 위치에 하나씩 배정됩니다. 같은 replica가 두 번 이상 선택되면 alert가 발생한다고 하겠습니다.

이 실습에서는 원시 로그를 먼저 세고, 무경보 로그의 보완사건으로 alert 확률을 계산합니다. 첫 exercise는 branch 수를 코드로 복원하고, 두 번째 exercise는 그 결과가 확률로 바뀌는 조건을 해석합니다.


In [1]:
# 공통 준비
import numpy as np


## E01. 원시 로그와 무경보 로그 세기

### 상황

`replica_count`개의 라벨된 replica가 있고, `positions`개의 순서 있는 요청 위치가 있습니다. 각 위치는 모든 replica 중 하나를 균등하게, 다른 위치와 독립적으로 고릅니다. 선택은 복원추출입니다.

무경보 로그는 모든 위치가 서로 다른 replica를 고른 로그입니다. 위치 수가 replica 수보다 크면 그런 로그는 없습니다.

### 구현할 계약

- `count_router_logs`는 모든 위치를 복원추출하는 원시 로그 수 `replica_count ** positions`를 첫 반환값으로 돌려야 합니다.
- `count_router_logs`는 서로 다른 replica만 쓴 무경보 로그 수를 둘째 반환값으로 돌리고, `positions > replica_count`이면 그 값은 `0`이어야 합니다.
- `replica_count`와 `positions` 중 하나라도 1보다 작으면 `ValueError`로 거부해야 합니다.

### 작은 예

4개 replica와 3개 위치라면 원시 로그 수는 `4 * 4 * 4`입니다. 무경보 로그는 첫 위치에서 4개, 둘째에서 3개, 셋째에서 2개의 선택이 가능한지 확인해 보세요.

<details><summary>힌트 1</summary>

원시 로그는 모든 위치의 branch 수가 같을 때의 곱입니다. 무경보 로그는 이미 쓴 replica를 다음 위치에서 제외합니다.

</details>

<details><summary>힌트 2</summary>

무경보 수는 `1`에서 시작해 각 위치에서 남은 선택 수를 차례로 곱하면 됩니다. 위치가 replica보다 많아지는 순간에는 바로 `0`이 됩니다.

</details>


In [5]:
import math


def count_router_logs(replica_count: int, positions: int) -> tuple[int, int]:
    """Return (all ordered logs, all-distinct no-alert logs)."""
    if replica_count < 1 or positions < 1:
        raise ValueError("replica_count and positions must both be at least 1")

    # TODO: 모든 위치가 독립적으로 고르는 원시 로그 수를 계산하세요.
    total_logs = replica_count ** positions

    # TODO: 이미 고른 replica를 제외한 무경보 로그 수를 계산하세요.
    no_alert_logs = math.perm(replica_count, positions)

    return int(total_logs), int(no_alert_logs)


In [6]:
sample_total, sample_no_alert = count_router_logs(4, 3)
print(f"원시 로그: {sample_total}, 무경보 로그: {sample_no_alert}")


원시 로그: 64, 무경보 로그: 24


In [7]:
def check_e01() -> None:
    total, no_alert = count_router_logs(4, 3)
    np.testing.assert_equal(total, 64)
    np.testing.assert_equal(no_alert, 24)

    total_one, no_alert_one = count_router_logs(4, 1)
    np.testing.assert_equal((total_one, no_alert_one), (4, 4))

    total_long, no_alert_long = count_router_logs(3, 4)
    np.testing.assert_equal(total_long, 81)
    np.testing.assert_equal(no_alert_long, 0)

    try:
        count_router_logs(0, 3)
    except ValueError:
        pass
    else:
        raise AssertionError("replica_count가 1 미만이면 ValueError여야 합니다")


check_e01()


### 확인 결과 정리

선택 복습입니다. 원시 로그와 무경보 로그에서 각 위치의 branch 수가 어떻게 달랐는지 메모해도 좋습니다. 이 메모는 실습 완료 조건이 아닙니다.


## E02. 보완사건으로 alert 확률 해석하기

### 상황

E01의 두 반환값은 같은 원시 표본공간을 셉니다. 따라서 무경보 로그의 비율을 먼저 구한 뒤, 전체에서 빼면 alert가 한 번 이상 생길 확률을 얻습니다.

### 구현할 계약

- `collision_probability`는 같은 원시 표본공간의 무경보 수를 사용해 `1.0 - no_alert_logs / total_logs`를 반환해야 합니다.
- `total_logs`가 양수가 아니거나 `no_alert_logs`가 0과 `total_logs` 사이가 아니면 `ValueError`로 거부해야 합니다.
- 해석에는 라벨된 원시 로그와 순서를 지운 보고를 구분하고, 라벨만으로 동등가능성이 생기지 않으며 각 위치의 균등·독립 선택이 원시 로그의 같은 확률을 정당화한다는 내용을 써야 합니다.

### 작은 예

E01에서 `(64, 24)`를 얻었다면, `24 / 64`는 무경보 확률입니다. alert 확률은 그 수 자체가 아니라 그 보완사건입니다.

<details><summary>힌트 1</summary>

`no_alert_logs / total_logs`가 무엇을 나타내는지 먼저 문장으로 말해 보세요. 그 사건의 보완사건이 alert입니다.

</details>

<details><summary>힌트 2</summary>

라벨은 `(R1, R2, R4)`처럼 원시 결과를 구분합니다. 원시 로그가 모두 같은 확률이 되려면 라벨 외에 각 위치의 선택이 균등하고 독립적이라는 조건이 필요합니다.

</details>


In [8]:
def collision_probability(total_logs: int, no_alert_logs: int) -> float:
    """Return the probability that at least one replica is selected twice."""
    if total_logs <= 0 or not 0 <= no_alert_logs <= total_logs:
        raise ValueError("counts must describe one nonempty sample space")

    probability = 1 - no_alert_logs / total_logs
    return float(probability)


In [9]:
print(f"alert 확률: {collision_probability(64, 24):.3f}")


alert 확률: 0.625


In [10]:
def check_e02() -> None:
    np.testing.assert_allclose(collision_probability(64, 24), 5 / 8)
    np.testing.assert_allclose(collision_probability(81, 0), 1.0)

    try:
        collision_probability(64, 65)
    except ValueError:
        pass
    else:
        raise AssertionError("no_alert_logs가 total_logs보다 크면 ValueError여야 합니다")


check_e02()


### 결과 해석

아래 빈칸에 2~4문장으로 적으세요. `(R1, R2, R4)` 같은 라벨된 원시 로그와 순서를 지운 보고가 어떻게 다른지, 그리고 왜 원시 로그를 같은 확률로 셀 수 있었는지를 함께 설명하세요.

<!-- TODO: 원시 로그와 동등가능성의 조건을 해석하세요. -->
(R1,R2,R4) 같은 원시 로그는 누가/몇 번째에 무엇이 선택됐는지까지 보존한 미시적 결과이고, 순서를 지운 {R1,R2,R4}는 여러 원시 로그를 하나로 합친 요약 결과다.
각 위치가 4개 replica 중 하나를 균등하고 독립적으로 고르므로, 모든 원시 로그의 확률은 1/64이다.
순서를 지운 보고는 서로 다른 개수의 원시 로그를 합칠 수 있으므로, 보고끼리는 자동으로 같은 확률이 아니다.
